# 01 — Dataset Audit
**Project:** Machine Learning-Based Prediction of Electrical Grid Stability (IT5022 Group Assignment)

**Goal:** confirm the dataset's schema, size, data quality and class balance before any modelling.

**Source:** Arzamasov, V. (2018). *Electrical Grid Stability Simulated Data* [Dataset]. UCI Machine
Learning Repository. https://doi.org/10.24432/C5PG66 — mirrored on Kaggle:
https://www.kaggle.com/datasets/avikdas567/electrical-grid-stability-and-control-simulation

In [1]:
import sys
from pathlib import Path

# Make the project's src/ package importable when running from notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "electrical_grid_stability.csv"

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 140)
%matplotlib inline

In [2]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (10000, 14)
Columns: ['tau1', 'tau2', 'tau3', 'tau4', 'p1', 'p2', 'p3', 'p4', 'g1', 'g2', 'g3', 'g4', 'stab', 'stabf']


,tau1,tau2,tau3,tau4,p1,p2,p3,p4,g1,g2,g3,g4,stab,stabf
0,2.959060,3.079885,8.381025,9.780754,3.763085,-0.782604,-1.257395,-1.723086,0.650456,0.859578,0.887445,0.958034,0.055347,unstable
1,9.304097,4.902524,3.047541,1.369357,5.067812,-1.940058,-1.872742,-1.255012,0.413441,0.862414,0.562139,0.781760,-0.005957,stable
2,8.971707,8.848428,3.046479,1.214518,3.405158,-1.207456,-1.277210,-0.920492,0.163041,0.766689,0.839444,0.109853,0.003471,unstable
3,0.716415,7.669600,4.486641,2.340563,3.963791,-1.027473,-1.938944,-0.997374,0.446209,0.976744,0.929381,0.362718,0.028871,unstable
4,3.134112,7.608772,4.943759,9.857573,3.525811,-1.125531,-1.845975,-0.554305,0.797110,0.455450,0.656947,0.820923,0.049860,unstable


In [3]:
df.dtypes.to_frame("dtype").T

,tau1,tau2,tau3,tau4,p1,p2,p3,p4,g1,g2,g3,g4,stab,stabf
dtype,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,str


All 12 operating features and `stab` are floating-point numbers; `stabf` is a text label. No type conversion is needed.

In [4]:
print("Missing values per column:")
print(df.isna().sum().to_frame("missing").T)
print("\nExact duplicate rows:", df.duplicated().sum())
print("Duplicate feature vectors (ignoring labels):", df.drop(columns=["stab", "stabf"]).duplicated().sum())

Missing values per column:
         tau1  tau2  tau3  tau4  p1  p2  p3  p4  g1  g2  g3  g4  stab  stabf
missing     0     0     0     0   0   0   0   0   0   0   0   0     0      0

Exact duplicate rows: 0
Duplicate feature vectors (ignoring labels): 0


**Interpretation:** there are no missing values and no duplicated records, so no imputation or
de-duplication is required. Cleansing therefore focuses on *validity* (are values physically
plausible?) and *leakage* (does any column reveal the label?).

In [5]:
from src.data_checks import validity_checks
validity_checks(df)

,Check,Violating rows,Max |p1+p2+p3+p4|
0,"tau1-tau4 within [0.5, 10]",0,NaN
1,"p2-p4 within [-2, -0.5]",0,NaN
2,"g1-g4 within [0.05, 1]",0,NaN
3,p1 + p2 + p3 + p4 = 0 (|sum| <= 1e-9),0,5.773160e-15
4,stabf == 'stable' exactly when stab <= 0,0,NaN
5,stabf only contains 'stable'/'unstable',0,NaN


**Interpretation:**
- Every feature lies inside the range documented for the simulation, so there are no invalid values or outliers to remove.
- The power balance `p1 + p2 + p3 + p4 = 0` holds to floating-point precision (≈ 6×10⁻¹⁵). `p1` is
  therefore an **exact linear function** of `p2`–`p4`: one of the 12 features is fully redundant. This
  matters for preprocessing and for Logistic Regression (see notebooks 03 and 07).
- `stabf` is `stable` **exactly** when `stab ≤ 0` — 0 violations out of 10,000. `stabf` is a
  deterministic threshold of `stab`, so using `stab` as an input would be target leakage.

In [6]:
counts = df["stabf"].value_counts()
pd.DataFrame({"count": counts, "percent": (counts / len(df) * 100).round(2)})

,count,percent
stabf,,
unstable,6380,63.8
stable,3620,36.2


**Interpretation:** the classes are moderately imbalanced (63.8% unstable / 36.2% stable). A
classifier that always predicts `unstable` already reaches **63.8% accuracy**, so this majority
baseline is the minimum any useful model must beat, and class-wise metrics (precision, recall,
F1, macro-F1) must be reported alongside accuracy.

In [7]:
df.describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
tau1,10000.0,5.250,2.743,0.501,2.875,5.250,7.625,9.999
tau2,10000.0,5.250,2.743,0.500,2.875,5.250,7.625,10.000
tau3,10000.0,5.250,2.743,0.501,2.876,5.250,7.625,9.999
tau4,10000.0,5.250,2.743,0.500,2.875,5.250,7.625,9.999
p1,10000.0,3.750,0.752,1.583,3.218,3.751,4.282,5.864
p2,10000.0,-1.250,0.433,-2.000,-1.625,-1.250,-0.875,-0.500
p3,10000.0,-1.250,0.433,-2.000,-1.625,-1.250,-0.875,-0.500
p4,10000.0,-1.250,0.433,-2.000,-1.625,-1.250,-0.875,-0.500
g1,10000.0,0.525,0.274,0.050,0.288,0.525,0.762,1.000
g2,10000.0,0.525,0.274,0.050,0.288,0.525,0.762,1.000


**Interpretation:** `tau1`–`tau4` span 0.5–10 s, `g1`–`g4` span 0.05–1, and `p2`–`p4` span −2 to −0.5,
each with almost identical mean and spread. This is consistent with the simulation drawing each
parameter independently and uniformly. `p1` (producer) is the negative sum of the three consumers,
which is why its range (1.58–5.86) differs.